In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/allstate-claims-severity/sample_submission.csv
/kaggle/input/allstate-claims-severity/train.csv.zip
/kaggle/input/allstate-claims-severity/sample_submission.csv.zip
/kaggle/input/allstate-claims-severity/train.csv
/kaggle/input/allstate-claims-severity/test.csv
/kaggle/input/allstate-claims-severity/test.csv.zip


---
title: "Allstate Claims Severity — EDA & LightGBM (tidymodels)"
author: "Your Name"
output: html_document
---

```{r setup, message=FALSE, warning=FALSE}
knitr::opts_chunk$set(echo = TRUE, message = FALSE, warning = FALSE)
```

## 1. Introduction

In this notebook I explore the **Allstate Claims Severity** dataset and build a **LightGBM** model using the **tidymodels** ecosystem.

Main steps:

1. Load libraries and data  
2. Explore the target variable `loss` and its log transform  
3. Explore continuous features (`cont*`)  
4. Explore categorical features (`cat*`)  
5. Examine relationships between features and `loss`  
6. Engineer features in a way that makes sense for boosted trees  
7. Build and tune a LightGBM model via `tune_race_anova`  
8. Train the final model and generate a Kaggle submission  

Throughout, I explain *why* each feature engineering step was chosen, not just *how* it was implemented.

---

## 2. Libraries & Data (EDA)

```{r libraries}
library(tidyverse)
library(DataExplorer)
library(GGally)
library(scales)
library(forcats)
library(vroom)

# Modeling libraries
library(rpart)
library(tidymodels)
library(embed)
library(bonsai)
library(lightgbm)
library(themis)
library(doFuture)
library(future)
library(finetune)
library(readr)
library(progressr)
library(lightgbm)   # loaded again is harmless
```

```{r data-load-eda}
# For EDA, use vroom to quickly read the training data
train <- vroom::vroom("train.csv")

glimpse(train)
```

---

## 3. Target Variable: `loss`

The target `loss` in this competition is a **positive, highly skewed** continuous variable.  
Models like gradient boosting can technically handle skewed targets, but:

- Large outliers can dominate the loss function  
- Optimization can become less stable  
- Errors on high-loss examples are often more important than small differences at the low end  

To help with this, we use a **log(1 + loss)** transform:

- `log1p(loss)` compresses large values and spreads out small ones  
- It tends to make the distribution closer to normal  
- Many models optimize more smoothly on this transformed scale  

Later, we’ll train the model on the log scale and back-transform with `exp(pred) - 1`.

```{r loss-summary}
summary(train$loss)

quantile(
  train$loss,
  probs = c(0, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 1),
  na.rm = TRUE
)

# Add a logged version of loss for EDA
train <- train %>%
  mutate(log_loss = log1p(loss))   # log(1 + loss)
```

We also quickly check for missingness across the dataset:

```{r missingness-plot}
plot_missing(train)
```

### 3.1 Distributions of `loss` and `log(1 + loss)`

```{r loss-distributions, fig.height=4, fig.width=7}
p_loss <- ggplot(train, aes(x = loss)) +
  geom_histogram(bins = 100) +
  scale_x_continuous(labels = scales::comma) +
  labs(
    title = "Distribution of loss",
    x = "Loss",
    y = "Count"
  ) +
  theme_minimal()

p_log_loss <- ggplot(train, aes(x = log_loss)) +
  geom_histogram(bins = 60) +
  labs(
    title = "Distribution of log(1 + loss)",
    x = "log(1 + loss)",
    y = "Count"
  ) +
  theme_minimal()

p_loss
p_log_loss
```

**Why this feature engineering step?**

- The original `loss` distribution has a long right tail.  
- On the log scale, the distribution is more symmetric and concentrated, which generally makes it easier for the model to learn patterns and avoid being overly driven by a few huge claims.  
- Kaggle solutions for this competition commonly model `log(loss)` or `log(1 + loss)` for this exact reason.

---

## 4. Continuous Features (`cont*`)

Allstate provides a set of anonymized continuous features `cont1`–`cont14`. We start by summarizing them.

```{r cont-summary}
cont_cols <- train %>%
  select(starts_with("cont")) %>%
  names()

train %>%
  select(all_of(cont_cols)) %>%
  summary()
```

### 4.1 Histograms of Continuous Features

```{r cont-histograms, fig.height=7, fig.width=10}
cont_long <- train %>%
  select(all_of(cont_cols)) %>%
  pivot_longer(cols = everything(),
               names_to = "feature",
               values_to = "value")

p_cont_hist <- ggplot(cont_long, aes(x = value)) +
  geom_histogram(bins = 30) +
  facet_wrap(~ feature, scales = "free") +
  labs(
    title = "Distributions of continuous features",
    x = "Value",
    y = "Count"
  ) +
  theme_minimal()

p_cont_hist
```

**Why not normalize/standardize these features?**

- For **tree-based models** like LightGBM, feature scaling (standardizing to mean 0, SD 1) is usually *not necessary*: trees split based on order and thresholds, not on magnitude.  
- The `cont` variables are already on relatively comparable scales (mostly within \[0, 1\] or similar).  
- As a result, we don’t add a `step_normalize()` step for the predictors—we keep them as-is and let the boosting model find the useful splits.

---

## 5. Correlation: Continuous Features and `log_loss`

We compute correlations between each continuous feature and `log_loss` and visualize the overall correlation structure.

```{r cont-correlations, fig.height=7, fig.width=7}
cont_corr_data <- train %>%
  select(all_of(c(cont_cols, "log_loss")))

cont_corr <- cor(cont_corr_data, use = "complete.obs")
print(round(cont_corr["log_loss", ], 3))

ggcorr(
  data = cont_corr_data,
  label = TRUE,
  label_round = 2,
  hjust = 0.9,
  size = 3
) +
  ggtitle("Correlation among continuous features and log_loss")
```

**Interpretation and motivation:**

- Most continuous features show **moderate or weak** correlation with `log_loss`.  
- That doesn’t mean they’re useless: tree-based models excel at capturing **interactions** and **non-linear relationships** that simple correlation doesn’t reveal.  
- The correlation analysis is more of a sanity check: no single continuous feature appears to completely dominate the signal.

---

## 6. Categorical Features (`cat*`)

This dataset includes many high-cardinality categorical features. We begin by counting the number of distinct levels per feature.

```{r cat-cardinality}
cat_cols <- train %>%
  select(starts_with("cat")) %>%
  names()

cat_card <- train %>%
  summarise(across(all_of(cat_cols), n_distinct)) %>%
  pivot_longer(cols = everything(),
               names_to = "feature",
               values_to = "n_levels") %>%
  arrange(desc(n_levels))

head(cat_card, 20)
summary(cat_card$n_levels)
```

### 6.1 Top 30 Categorical Features by Number of Levels

```{r cat-card-plot, fig.height=7, fig.width=7}
p_cat_card <- cat_card %>%
  slice_max(order_by = n_levels, n = 30) %>%
  ggplot(aes(x = reorder(feature, n_levels), y = n_levels)) +
  geom_col() +
  coord_flip() +
  labs(
    title = "Number of levels (top 30 categorical features)",
    x = "Categorical feature",
    y = "Number of levels"
  ) +
  theme_minimal()

p_cat_card
```

**Why this matters for feature engineering:**

- Many categorical variables have **dozens or even hundreds** of levels.  
- Naive **one-hot encoding** would explode the dimensionality and lead to very sparse matrices, which:  
  - Slows down training and increases memory usage  
  - Can encourage overfitting to tiny groups of observations  
- Instead, we’ll:  
  1. **Collapse very rare categories** into an `"other"` level.  
  2. Use **target (likelihood) encoding** to turn categories into numeric features that reflect their relationship with `loss`.

---

### 6.2 Example: Mean Loss by Category Level

```{r plot-cat-vs-loss}
plot_cat_vs_loss <- function(df, var, min_count = 100, top_n = 20) {
  df %>%
    group_by(.data[[var]]) %>%
    summarise(
      mean_loss   = mean(loss),
      median_loss = median(loss),
      n           = n(),
      .groups     = "drop"
    ) %>%
    filter(n >= min_count) %>%                 # drop super-rare levels
    arrange(desc(mean_loss)) %>%
    slice_head(n = top_n) %>%
    mutate(level = fct_reorder(as.factor(.data[[var]]), mean_loss)) %>%
    ggplot(aes(x = level, y = mean_loss)) +
      geom_col() +
      coord_flip() +
      labs(
        title = paste("Mean loss by", var),
        x = var,
        y = "Mean loss"
      ) +
      theme_minimal()
}

example_cats <- c("cat1", "cat2", "cat3", "cat116")

for (v in example_cats) {
  if (v %in% names(train)) {
    print(plot_cat_vs_loss(train, v, min_count = 100, top_n = 20))
  }
}
```

**Why plot this?**

- These plots show that different category levels can have **very different average claim severities**.  
- That’s exactly what target encoding will try to summarize numerically: it maps each category to a value that reflects its typical loss, while controlling for overfitting.

---

## 7. Relationships Between Top Continuous Features and `log_loss`

We identify the continuous features most correlated with `log_loss` and inspect scatter/smoother plots.

```{r top-cont-vs-logloss, fig.height=6, fig.width=8}
cont_loss_corr <- train %>%
  select(all_of(c(cont_cols, "log_loss"))) %>%
  cor(use = "complete.obs") %>%
  .["log_loss", cont_cols] %>%
  sort(decreasing = TRUE)

print(cont_loss_corr)

top_conts <- names(cont_loss_corr)[1:4]

for (v in top_conts) {
  p <- ggplot(train, aes(x = .data[[v]], y = log_loss)) +
    geom_point(alpha = 0.1) +
    geom_smooth(se = FALSE) +
    labs(
      title = paste("log(1 + loss) vs", v),
      x = v,
      y = "log(1 + loss)"
    ) +
    theme_minimal()
  print(p)
}
```

**Why this step?**

- Even if we don’t explicitly hand-craft non-linear features, these plots show **how** each continuous feature relates to the target (monotonic, U-shaped, etc.).  
- It confirms that there is signal to exploit and that a flexible model like LightGBM (with many small trees) can pick up on these shapes without us explicitly creating polynomial or spline terms.

---

## 8. Modeling: LightGBM with tidymodels

Now we move from EDA into modeling. The key feature engineering decisions implemented in the **recipe** are:

1. Treat `id` as an identifier only (not a predictor)  
2. Collapse rare categorical levels into `"other"`  
3. Apply **likelihood encoding** (`step_lencode`) to high-cardinality categorical variables  
4. Log-transform the outcome `loss` with `log(1 + loss)`  
5. Remove any predictors with zero variance  

We combine these with a LightGBM model via **bonsai** and tune hyperparameters with `tune_race_anova`.

### 8.1 Parallel & Environment Setup

```{r parallel-setup}
set.seed(123)

# Prevent BLAS libraries from oversubscribing cores
Sys.setenv(
  OMP_NUM_THREADS = "1",
  MKL_NUM_THREADS = "1",
  OPENBLAS_NUM_THREADS = "1",
  NUMEXPR_NUM_THREADS = "1"
)

# Allow larger globals for future (helps with racing tuner)
Sys.setenv(R_FUTURE_GLOBALS_MAXSIZE = "Inf")
options(
  future.scheduling      = 2,
  future.globals.maxSize = +Inf
)

plan(multisession, workers = 4)
registerDoFuture()
```

### 8.2 Reload Data for Modeling

For modeling we reload using `readr::read_csv()` for consistency with other tidymodels examples.

```{r reload-data}
train <- readr::read_csv("train.csv")
test  <- readr::read_csv("test.csv")
ss    <- readr::read_csv("sample_submission.csv")

glimpse(train)
```

### 8.3 Model Specification (LightGBM via bonsai)

```{r model-spec}
boost_model <- boost_tree(
  tree_depth = tune(),
  trees      = tune(),
  learn_rate = tune()
) %>%
  set_engine("lightgbm") %>%
  set_mode("regression")
```

**Why LightGBM?**

- Gradient-boosted trees are a standard strong baseline for tabular Kaggle competitions.  
- LightGBM in particular is optimized for speed and performance on large, sparse, or high-cardinality datasets.  
- It handles non-linearities and interactions automatically, which is perfect given the rich mix of `cat*` and `cont*` features.

---

### 8.4 Recipe (Feature Engineering for Modeling)

This is where we explicitly encode the “feature engineering logic” we want for the model.

```{r recipe}
claims_rec <- recipe(loss ~ ., data = train) %>%
  # 1. `id` is just an identifier, not a predictive feature
  update_role(id, new_role = "ID") %>%
  
  # 2. Collapse rare categories into "other"
  step_other(all_nominal_predictors(), threshold = 0.01, other = "other") %>%
  
  # 3. Likelihood (target) encoding of categorical predictors based on `loss`
  step_lencode(all_nominal_predictors(), outcome = vars(loss)) %>%
  
  # 4. Log-transform the outcome for more stable modeling, but skip when computing metrics
  step_log(loss, offset = 1, skip = TRUE) %>%  # model log(1 + loss)
  
  # 5. Drop zero-variance predictors that carry no information
  step_zv(all_predictors())
```

**Why each step?**

- **`update_role(id, new_role = "ID")`**  
  - The `id` column is just an identifier for each row.  
  - Using it as a predictor would leak arbitrary numbers into the model that have no real meaning and can encourage overfitting.  
  - Changing its role keeps it in the data for later use (e.g., joining predictions) without letting the model see it.

- **`step_other(all_nominal_predictors(), threshold = 0.01, other = "other")`**  
  - Many categorical features have levels that appear only a handful of times.  
  - Those levels are too rare for the model to learn stable patterns about them.  
  - Grouping them into an `"other"` bucket reduces noise and lowers the risk of overfitting to tiny groups.

- **`step_lencode(all_nominal_predictors(), outcome = vars(loss))`**  
  - Instead of one-hot encoding, we use **likelihood/target encoding**.  
  - Roughly, it replaces each category level with a numeric value summarizing how that
